In [ ]:
from google.cloud import bigquery
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
import warnings
warnings.filterwarnings(action='ignore', message="Your application has authenticated using end user")

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
# Add the name of the project in google console
client = bigquery.Client(project="citric-trees-489611-k5")

# Data Preparation - Machine Measurements

In [ ]:
machine_measurements_query = """
SELECT
    es.subject_id,
    es.stay_id,
    es.hadm_id,
    mm.ecg_time,
    mm.report_0,
    mm.report_1,
    mm.report_2,
    mm.report_3,
    mm.report_4,
    mm.report_5,
    mm.report_6,
    mm.report_7,
    mm.report_8,
    mm.report_9,
    mm.report_10,
    mm.report_11,
    mm.report_12,
    mm.report_13,
    mm.report_14,
    mm.report_15,
    mm.report_16,
    mm.report_17

FROM physionet-data.mimiciv_ed.edstays es

INNER JOIN physionet-data.mimiciv_ecg.machine_measurements mm
    ON mm.subject_id = es.subject_id
    -- restrict to ECGs recorded during the ED stay window
    AND mm.ecg_time >= es.intime
    AND mm.ecg_time <= es.outtime
"""

machine_measurements_df = client.query(machine_measurements_query).to_dataframe()

print(machine_measurements_df.shape)
print(machine_measurements_df.dtypes)
print(f"Unique ED stays: {machine_measurements_df['stay_id'].nunique()}")
print(f"Unique subjects: {machine_measurements_df['subject_id'].nunique()}")

machine_measurements_df["ecg_time"] = pd.to_datetime(
    machine_measurements_df["ecg_time"], utc=True
)

In [ ]:
# Define regex patterns
# Each report_i column is an independent ECG machine finding.
# We scan each column separately to preserve finding count and avoid
# cross-column boundary artifacts from concatenation.

negative_patterns = re.compile(r"""
    ACUTE\s+INFARCT|
    INFARCT(?:ION)?|
    ISCH[EAE]MIA|
    MYOCARDIAL\s+INJURY|
    STRONGLY\s+SUGGESTS|
    CONSIDER\s+ACUTE|
    POSSIBLY\s+ACUTE|
    PROBABLY\s+ACUTE|
    COMPLETE\s+AV\s+BLOCK|
    LEFT\s+BUNDLE\s+BRANCH\s+BLOCK|
    RIGHT\s+BUNDLE\s+BRANCH\s+BLOCK|
    VENTRICULAR\s+TACHYCARDIA|
    ATRIAL\s+FIBRILLATION|
    ATRIAL\s+FLUTTER|
    VENTRICULAR\s+HYPERTROPHY|
    PROLONGED\s+QT|
    LONG\s+QTc|
    PERICARDITIS|
    ST\s+ELEVATION.*INFARCT|
    WIDE[\s\-]QRS\s+TACHYCARDIA|
    COMPLETE\s+HEART\s+BLOCK|
    EXTREME\s+TACHYCARDIA|
    VENTRICULAR\s+PREEXCITATION|
    WPW\s+PATTERN
""", re.IGNORECASE | re.VERBOSE)

neutral_patterns = re.compile(r"""
    BORDERLINE|
    POSSIBLE(?!\s+ACUTE)|
    PROBABLE(?!\s+ACUTE)|
    NONSPECIFIC|
    NON[\s\-]SPECIFIC|
    CANNOT\s+RULE\s+OUT|
    MAY\s+BE\s+DUE|
    EQUIVOCAL|
    ABNORMAL\s+FOR\s+AGE|
    CONSIDER(?!\s+ACUTE)|
    AGE\s+UNDETERMINED|
    AGE\s+INDETERMINATE|
    ST[\s\-]T\s+CHANGES|
    T\s+WAVE\s+CHANGES|
    REPOLARIZATION\s+ABNORMALITY|
    CONDUCTION\s+DEFECT|
    AXIS\s+DEVIATION|
    ATRIAL\s+ABNORMALITY|
    ATRIAL\s+ENLARGEMENT|
    LOW\s+QRS\s+VOLTAGE|
    POOR\s+R\s+WAVE\s+PROGRESSION|
    ALL\s+12\s+LEADS\s+ARE\s+MISSING|
    NOT\s+ENOUGH\s+LEADS|
    DATA\s+QUALITY|
    UNSUITABLE\s+FOR\s+ANALYSIS|
    RECORDING\s+UNSUITABLE|
    TECHNICAL\s+ERROR|
    ANALYSIS\s+ERROR|
    PEDIATRIC|
    MEASUREMENT\s+ERROR|
    NO\s+FURTHER\s+ANALYSIS
""", re.IGNORECASE | re.VERBOSE)

# Anchored to full cell value — strip() is called before matching,
# so ^ and $ reliably catch cells that are purely these phrases
normal_patterns = re.compile(r"""
    ^\s*NORMAL\s+ECG\s*$|
    WITHIN\s+NORMAL\s+LIMITS|
    NORMAL\s+VARIANT|
    AVAILABLE\s+LEADS\s+NORMAL|
    NORMAL\s+ECG\s+BASED\s+ON|
    NORMAL\s+ECG\s+EXCEPT\s+FOR\s+RATE|
    ^SINUS\s+RHYTHM\s*\.?\s*$
""", re.IGNORECASE | re.VERBOSE)

In [ ]:
# Classify each cell
# Returns a numeric score directly: 0=normal, 1=neutral, 2=abnormal, -1=empty
# Priority: abnormal > neutral > normal > empty
def classify_cell(value):
    if value is None or str(value).strip() in ["nan", "", "None"]:
        return -1   # sentinel for empty — handled after row-level max
    v = str(value).strip()
    if negative_patterns.search(v):
        return 2    # abnormal
    elif neutral_patterns.search(v):
        return 1    # neutral/unknown
    elif normal_patterns.search(v):
        return 0    # normal
    else:
        return 1    # unrecognized → neutral, not normal

In [ ]:
# Score all 18 columns and take the row-level max
report_cols = [f"report_{i}" for i in range(18)]
score_cols  = [f"report_{i}_score" for i in range(18)]

for report_col, score_col in zip(report_cols, score_cols):
    machine_measurements_df[score_col] = (
        machine_measurements_df[report_col].apply(classify_cell)
    )

# Row-level max across all 18 score columns
# A patient with any abnormal finding (2) gets 2; any neutral gets 1; all normal gets 0
machine_measurements_df["ecg_acuity"] = (
    machine_measurements_df[score_cols].max(axis=1)
)

In [ ]:
# Handle no-ECG patients
# If all 18 columns were empty, max score will be -1 (the sentinel).
# Map these to 1 (neutral/unknown) — a missing ECG is not the same as a normal one.
machine_measurements_df["ecg_acuity"] = (
    machine_measurements_df["ecg_acuity"]
    .replace(-1, 1)
    .astype(int)
)

In [ ]:
# Resolve multiple ECGs per ED stay
# If a patient had more than one ECG during their ED stay, keep the row
# with the highest acuity. Ties are broken by earliest ecg_time — the first
# abnormal finding during the stay is the most clinically conservative choice.
machine_measurements_df = (
    machine_measurements_df
    .sort_values(["stay_id", "ecg_acuity", "ecg_time"],
                 ascending=[True, False, True])
    .drop_duplicates(subset="stay_id", keep="first")
    .reset_index(drop=True)
)

In [ ]:
# Final table
ecg_summary_df = machine_measurements_df[[
    "subject_id", "stay_id", "hadm_id", "ecg_time", "ecg_acuity"
]].copy()

print(ecg_summary_df["ecg_acuity"].value_counts().sort_index())
print(f"\nShape: {ecg_summary_df.shape}")
print(ecg_summary_df.head())

ecg_summary_df.to_csv("ecg_features.csv", index=False)

In [ ]:
from huggingface_hub import login
login('')

In [ ]:
from datasets import Dataset, DatasetInfo

PHYSIONET_LICENSE = "PhysioNet Credentialed Health Data License 1.5.0 — https://physionet.org/content/mimiciv/view-license/3.1/"

ecg_info = DatasetInfo(
    description=(
        "ECG acuity labels derived from the MIMIC-IV ECG machine_measurements table. "
        "One row per patient (subject_id). "
        "Each of the 18 report columns was classified independently using regex pattern "
        "matching with priority order: abnormal > neutral > normal > empty. "
        "ecg_acuity reflects the highest severity finding observed across all 18 report "
        "columns: 0=normal, 1=neutral/unknown, 2=abnormal. "
        "Patients with all 18 report columns empty (no ECG conducted or recorded) "
        "are assigned ecg_acuity=1 (neutral/unknown) rather than 0, as absence of "
        "an ECG record is not equivalent to a normal ECG result."
    ),
    license=PHYSIONET_LICENSE,
)

ds_ecg = Dataset.from_pandas(ecg_summary_df, split='ecg_features', info=ecg_info)
ds_ecg.push_to_hub("ADS599-Capstone/modeling_data", config_name="ecg_features", data_dir="modeling_data")
print("Pushed ecg_features to HuggingFace Hub.")